# M1_210210 emission-line comparison

- Two arms use the same seed and current defaults.
- Fit figures remain in each executed fit notebook.

In [1]:
import os
from pathlib import Path
os.environ["JAX_PLATFORMS"] = "cpu"
import h5py
import numpy as np
import pandas as pd
from scipy.special import softmax
from astropy import units as u
from specutils.utils.wcs_utils import air_to_vac
import sys
ROOT = Path.cwd()
RESULTS = ROOT / "results/emission-line-marginalisation"
TARGET = "210210-M1_210210"
sys.path.insert(0, str(ROOT / "scripts"))
from build_dr2_quiescent_summary import FEH_OFFSET
from per_galaxy_diagnostics import weighted_quantile

def text(value):
    return value.decode() if isinstance(value, bytes) else str(value)

def load_arm(arm):
    folder = RESULTS / arm / TARGET
    out = {"folder": folder}
    with h5py.File(folder / "ceridwen_derived_outputs.h5") as f:
        names = [text(x) for x in f["summary/parameter"][:]]
        out["summary"] = pd.DataFrame({k: f[f"summary/{k}"][:] for k in ("q16", "q50", "q84")}, index=names)
        out["spectrum"] = {k: f["spectrum"][k][:] for k in f["spectrum"]}
        out["photometry"] = {k: f["photometry"][k][:] for k in f["photometry"]}
        out["redshift"] = float(f.attrs["redshift"])
        out["photometry_source"] = text(f.attrs["photometry_source"])
    with h5py.File(folder / "ceridwen_result.h5") as f:
        samples = f["samples"]
        out["samples"] = {k: samples[k][:] for k in samples if k not in ("log_weights", "log_likelihoods", "log_likelihoods_birth")}
        out["weights"] = softmax(samples["log_weights"][:])
        out["sampler_wall_s"] = float(samples.attrs["wall_time_s"])
        out["likelihood_calls"] = int(samples.attrs["n_likelihood_calls"])
        out["seed"] = int(f["model"].attrs["random_seed"])
    return out

arms = {name: load_arm(name) for name in ("eline_off", "eline_on")}
assert arms["eline_off"]["seed"] == arms["eline_on"]["seed"] == 20260832
assert arms["eline_off"]["photometry_source"] == arms["eline_on"]["photometry_source"] == "cosmos2025"

parameters = [
    ("mass-weighted age [Gyr]", "mass-weighted age [Gyr]", 0.0),
    ("[Fe/H] [dex]", "Z", FEH_OFFSET),
    ("[alpha/Fe] [dex]", "afe", 0.0),
    ("log10(M*/Msun)", "logmass", 0.0),
    ("tau_dust", "diffuse_tau_kc", 0.0),
    ("dust index", "diffuse_dust_index", 0.0),
]
def quantiles(arm, key, offset):
    a = arms[arm]
    if key in a["summary"].index:
        return a["summary"].loc[key, ["q16", "q50", "q84"]].to_numpy(dtype=float) + offset
    return np.array([weighted_quantile(np.ravel(a["samples"][key]), a["weights"], q) for q in (0.16, 0.5, 0.84)]) + offset
rows = []
for label, key, offset in parameters:
    o = quantiles("eline_off", key, offset)
    n = quantiles("eline_on", key, offset)
    rows.append({"parameter": label, "off_q16": o[0], "off_q50": o[1], "off_q84": o[2],
                 "on_q16": n[0], "on_q50": n[1], "on_q84": n[2], "on_minus_off": n[1]-o[1]})
comparison = pd.DataFrame(rows)
comparison.to_csv(RESULTS / "comparison.csv", index=False)
display(comparison)

from json import loads
manifest = loads((RESULTS / "arms_manifest.json").read_text())
runtime = pd.DataFrame([{"arm": name, "arm_wall_s": manifest[f"{name}/M1_210210"]["wall_s"],
                         "sampler_wall_s": a["sampler_wall_s"], "likelihood_calls": a["likelihood_calls"]}
                        for name, a in arms.items()])
display(runtime)

s_on = arms["eline_on"]["spectrum"]
wave = s_on["wavelength"]
centre = air_to_vac(4861.3 * u.AA).to_value(u.AA) * (1 + arms["eline_on"]["redshift"])
velocity = (wave / centre - 1) * 299792.458
rows = []
for half_width in (300, 1500):
    pixels = s_on["mask"].astype(bool) & (np.abs(velocity) <= half_width)
    for name, a in arms.items():
        s = a["spectrum"]
        pull = (s["observed"] - s["posterior_q50"]) / s["effective_uncertainty"]
        rows.append({"arm": name, "Hbeta_half_width_km_s": half_width, "pixels": int(pixels.sum()),
                     "mean_pull": float(np.mean(pull[pixels])), "rms_pull": float(np.sqrt(np.mean(pull[pixels]**2)))})
residuals = pd.DataFrame(rows)
residuals.to_csv(RESULTS / "hbeta-residuals.csv", index=False)
display(residuals)

,parameter,off_q16,off_q50,off_q84,on_q16,on_q50,on_q84,on_minus_off
0,mass-weighted age [Gyr],4.946654,5.074722,5.187970,4.516973,4.666978,4.854585,-0.407743
1,[Fe/H] [dex],-0.208077,-0.181462,-0.165637,-0.179254,-0.158387,-0.137877,0.023076
2,[alpha/Fe] [dex],0.040770,0.050305,0.060819,0.037075,0.047533,0.057945,-0.002772
3,log10(M*/Msun),11.577024,11.587496,11.598375,11.523619,11.536276,11.552343,-0.051220
4,tau_dust,0.341967,0.352889,0.364412,0.303275,0.317617,0.331772,-0.035272
5,dust index,-0.998430,-0.993958,-0.984409,-0.997204,-0.988876,-0.971741,0.005083


,arm,arm_wall_s,sampler_wall_s,likelihood_calls
0,eline_off,1898.3,891.459806,8424501
1,eline_on,4321.6,3275.683024,6145311


,arm,Hbeta_half_width_km_s,pixels,mean_pull,rms_pull
0,eline_off,300,26,0.334000,0.568436
1,eline_on,300,26,-0.049826,0.484330
2,eline_off,1500,134,0.213739,0.558720
3,eline_on,1500,134,0.132474,0.560548


## Line fluxes

- Rebuild the saved line-on model without sampling again.
- Draw line fluxes from the calibration posterior.

In [2]:
import contextlib
import io
import json
import warnings
import matplotlib
matplotlib.use("Agg", force=True)
import jax
import jax.numpy as jnp
from ceridwen.fit import load_result_h5

source = json.loads((arms["eline_on"]["folder"] / "M1_210210_executed.ipynb").read_text())
ns = {}
os.environ["CERIDWEN_TARGET_ID"] = "M1_210210"
os.environ["CERIDWEN_RANDOM_SEED"] = "20260832"
os.environ["CERIDWEN_RESULT_DIR"] = str(arms["eline_on"]["folder"])
os.environ["SPS_HOME"] = str(ROOT / "external/fsps")
with warnings.catch_warnings(), contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
    warnings.simplefilter("ignore")
    exec("".join(source["cells"][2]["source"]), ns)
    for index in (4, 6, 8):
        exec("".join(source["cells"][index]["source"]), ns)
    setup = "".join(source["cells"][10]["source"]).split("joint_result = run_sampler(", 1)[0]
    exec(setup, ns)

result = load_result_h5(arms["eline_on"]["folder"] / "ceridwen_result.h5")
weights = softmax(np.asarray(result.log_weights))
posterior_indices = np.random.default_rng(20260833).choice(len(weights), size=2000, replace=True, p=weights)
selected = {name: np.asarray(values)[posterior_indices] for name, values in result.samples.items()}
indices = np.linspace(0, 1999, 200, dtype=int)
model = ns["joint_model"]
batch = {}
for name, template in model.theta_init.items():
    values = selected[name][indices]
    if np.shape(template) == (1,) and values.ndim == 1:
        values = values[:, None]
    batch[name] = jnp.asarray(values)
predictions = model.predict_vmap(batch)
spectrum = np.asarray(predictions["spectrum"])
photometry = np.asarray(predictions["photometry"])
sigma = np.hypot(ns["spectrum_uncertainty"][None, :],
                 np.exp(np.asarray(batch["log_f_calib"]).reshape(-1, 1)) * np.abs(spectrum))
lines = ns["emission_line_columns"]
_, free_flux, _ = ns["calibration_polynomial"].posterior_draws_with_lines(
    ns["spectrum_flux"], spectrum, sigma, jax.vmap(lines.columns)(batch), ns["spectrum_mask"],
    jax.random.PRNGKey(20260834), ridge=lines.ridge,
    photometry=(lines.band_matrix, ns["phot_flux"], photometry,
                np.broadcast_to(ns["phot_uncertainty"], photometry.shape), ns["phot_fit_mask"]),
)
tie = np.eye(len(lines.names)) if lines.tie is None else np.asarray(lines.tie)
raw_flux = np.asarray(free_flux) @ tie.T
q16, q50, q84 = np.percentile(raw_flux * 1e18, [16, 50, 84], axis=0)
line_fluxes = pd.DataFrame({"line": lines.names, "rest_vacuum_A": lines.wave_rest,
                            "q16_1e-18_cgs": q16, "q50_1e-18_cgs": q50, "q84_1e-18_cgs": q84})
assert len(line_fluxes) == 23 and raw_flux.shape == (200, 23)
assert np.all(raw_flux >= 0)
i_4959 = lines.names.index("[O III] 4959")
i_5007 = lines.names.index("[O III] 5007")
assert np.allclose(raw_flux[:, i_5007] / raw_flux[:, i_4959], 3.010, rtol=1e-3)
line_fluxes.to_csv(RESULTS / "line-fluxes.csv", index=False)
display(line_fluxes)

,line,rest_vacuum_A,q16_1e-18_cgs,q50_1e-18_cgs,q84_1e-18_cgs
0,Ba-8 3798,3799.0277,4.583259,5.557406,6.630364
1,Ba-7 3835,3836.5280,5.338498,6.339264,7.242056
2,[Ne III] 3869,3869.9172,0.149492,0.513254,1.180798
3,He I 3888.63A,3889.7926,0.357124,1.672590,3.984919
4,Ba-6 3889,3890.2127,4.161857,6.504677,8.205094
5,[Ne III] 3968,3968.6543,0.045055,0.154688,0.355876
6,Ba-5 3970,3971.2551,10.093371,11.099005,12.245402
7,[S II] 4070,4069.8122,5.496799,6.690332,8.078709
8,[S II] 4078,4077.5644,0.094585,0.443016,0.988851
9,Ba-delta 4101.76A,4102.9514,1.161925,2.248782,3.632979
